In [80]:
!pip install -q "protobuf>=5.29.6" -U vllm "mistral_common>=1.8.6" loguru tqdm huggingface_hub python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 978.0 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 174.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 7.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 13.0 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xformers 0.0.32.post1 requires torch==2.8.0, but you have torch 2.10.0 which is incompatible.


In [81]:
!nvidia-smi

Thu Mar 12 12:39:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          Off |   00000000:04:00.0 Off |                    0 |
| N/A   34C    P0             69W /  700W |       4MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [82]:
from huggingface_hub import login
import os
import dotenv
dotenv.load_dotenv()
# Paste your HF token here (or set the HF_TOKEN env var in Colab Secrets)
HF_TOKEN = os.getenv("HF_TOKEN")

login(token=HF_TOKEN)
print("Logged in to HuggingFace ✓")

Logged in to HuggingFace ✓


In [83]:
from pathlib import Path
from huggingface_hub import snapshot_download

model_name = "mistralai/Devstral-Small-2-24B-Instruct-2512"

model_path = "/content/models/Devstral-Small-2-24B"

if Path(model_path).exists() and any(Path(model_path).iterdir()):
    print(f"Model already downloaded at {model_path} — skipping download ✓")
else:
    print(f"Downloading {model_name} → {model_path}")
    snapshot_download(
        repo_id=model_name,
        local_dir=model_path,
        token=HF_TOKEN,
        ignore_patterns=["*.md", "*.txt", "original/*"],  # skip docs/checkpoints
    )
    print(f"Saved to {model_path}")
    

Model already downloaded at /content/models/Devstral-Small-2-24B — skipping download ✓


In [84]:
CONFIG = {
    "model_path": model_path,
    "gpus": 1,
    "max_num_seqs": 8,
    "gpu_memory_utilization": 0.95,
    "temperature": 0.0,
    "max_total_tokens": 8192,
    "max_new_tokens": 512,
    "repeat": 1,
    "testset_path": "/content/decompile_eval.json",
    "output_path": "./output.json",
    "num_workers": 4,
}
os.environ["TOKENIZERS_PARALLELISM"] = "true"

print("Config loaded")


Config loaded


In [85]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import subprocess
import json
import traceback
import tempfile
import multiprocessing
import sys

from pathlib import Path
from tqdm import tqdm
from loguru import logger
from transformers import PreTrainedTokenizerFast
from vllm import LLM, SamplingParams

logger.add(sys.stdout, colorize=False, format="{time} {level} {message}")

def evaluate_func(params):
    c_func = params["c_func"]
    c_test = params["c_test"]
    c_func_decompile = params["c_func_decompile"]
    
    timeout = 10
    flag_compile = 0
    flag_run = 0
    c_include = ""
    
    for line in c_func.split("\n"):
        if "#include" in line:
            c_include += line + "\n"
            c_func = c_func.replace(line, "")
    for line in c_test.split("\n"):
        if "#include" in line:
            c_include += line + "\n"
            c_test = c_test.replace(line, "")

    c_combine = c_include + "\n" + c_func_decompile + "\n" + c_test
    c_onlyfunc = c_include + "\n" + c_func_decompile

    with tempfile.TemporaryDirectory() as temp_dir:
        pid = os.getpid()
        c_file = os.path.join(temp_dir, f"combine_{pid}.c")
        executable = os.path.join(temp_dir, f"combine_{pid}")
        c_file_onlyfunc = os.path.join(temp_dir, f"onlyfunc_{pid}.c")
        executable_onlyfunc = os.path.join(temp_dir, f"onlyfunc_{pid}")

        with open(c_file, "w") as f:
            f.write(c_combine)
        with open(c_file_onlyfunc, "w") as f:
            f.write(c_onlyfunc)
        
        try:
            subprocess.run(
                ["gcc", "-S", c_file_onlyfunc, "-o", executable_onlyfunc, "-lm"],
                check=True,
                timeout=timeout,
            )
            flag_compile = 1
        except:
            return flag_compile, flag_run
        
        try:
            subprocess.run(
                ["gcc", c_file, "-o", executable, "-lm"],
                check=True, timeout=timeout, capture_output=True,
            )
        except Exception:
            return flag_compile, flag_run
        
        try:
            subprocess.run(
                [executable], capture_output=True, text=True, timeout=timeout, check=True
            )
            flag_compile = 1
        except Exception:
            return flag_compile, flag_run
        
        try: 
            subprocess.run(
                [executable], capture_output=True, text=True,
                timeout=timeout, check=True
            )
            flag_run = 1
        except Exception:
            return flag_compile, flag_run
    
    return flag_compile, flag_run


In [90]:
def decompile_pass_rate(testsets, gen_results_repeat, opts, num_workers):
    all_stats = []

    for gen_results in gen_results_repeat:
        ctx = multiprocessing.get_context("fork")
        tasks = [
            {
                "c_func": testset["c_func"],
                "c_test": testset["c_test"],
                "c_func_decompile": output[0],
            }
            for testset, output in zip(testsets,gen_results)
        ]

        with ctx.Pool(num_workers) as pool:
            eval_results = list(tqdm(pool.imap(evaluate_func, tasks), total=len(tasks)))
        
        stats = {opt: {"compile": 0, "run": 0, "total": 0} for opt in opts}
        for testset, output, flag in tqdm(
            zip(testsets, gen_results, eval_results),
            total=len(testsets),
            desc="Tallying",
        ):
            opt = testset["type"]
            flag_compile, flag_run = flag
            stats[opt]["total"] += 1
            if flag_compile:
                stats[opt]["compile"] += 1
            if flag_run:
                stats[opt]["run"] += 1
        all_stats.append(stats)
    
    avg_stats = {opt: {"compile":0, "run": 0, "total": 0} for opt in opts}
    for stats in all_stats:
        for opt in opts:
            avg_stats[opt]["compile"] += stats[opt]["compile"]
            avg_stats[opt]["run"] += stats[opt]["run"]
            avg_stats[opt]["total"] += stats[opt]["total"]
        for opt, data in avg_stats.items():
            compile_rate = data["compile"] / data["total"] if data["total"] > 0 else 0
            run_rate = data["run"] / data["total"] if data["total"] > 0 else 0
            print(f"  Opt {opt}: Compile {compile_rate:.2%}  |  Run {run_rate:.2%}")

    return avg_stats
print("Functions defined ✓")


Functions defined ✓


In [96]:
def run_eval_pipeline(cfg: dict):
    testset_path = cfg["testset_path"]
    output_path  = cfg["output_path"]

    if not Path(testset_path).exists():
        raise ValueError(f"Testset not found: {testset_path}")

    with open(testset_path, "r") as f:
        testsets = json.load(f)
    logger.info(f"Loaded {len(testsets)} test cases")

    opts = {
    k: f"[INST] Decompile the following {k}-optimized assembly to C. Output only valid C code, no explanations.\n\nAssembly:\n"
    for k in ("O0", "O1", "O2", "O3")}
    after = "\n[/INST]\n"
    inputs = [opts[t["type"]] + t["input_asm_prompt"] + after for t in testsets]

    # Read eos_token directly from JSON — no transformers tokenizer needed
    with open(os.path.join(cfg["model_path"], "tokenizer_config.json")) as f:
        tok_cfg = json.load(f)
    eos_token = tok_cfg.get("eos_token", "</s>")
    stop_sequences = [eos_token]

    logger.info(f"Loading model: {cfg['model_path']}")
    llm = LLM(
        model=cfg["model_path"],
        tensor_parallel_size=cfg["gpus"],
        max_model_len=cfg["max_total_tokens"],
        gpu_memory_utilization=cfg["gpu_memory_utilization"],
    )

    sampling_params = SamplingParams(
        temperature=cfg["temperature"],
        max_tokens=cfg["max_new_tokens"],
        stop=stop_sequences,
    )

    gen_results_repeat = []
    for i in range(cfg["repeat"]):
        logger.info(f"Generation pass {i+1}/{cfg['repeat']}")
        raw = llm.generate(inputs, sampling_params)
        gen_results_repeat.append([[out.outputs[0].text] for out in raw])

    if output_path:
        save_data = []
        for testset, res in zip(testsets, gen_results_repeat[0]):
            testset["output"] = res[0]
            save_data.append(testset)
        with open(output_path, "w") as f:
            json.dump(save_data, f, indent=4, ensure_ascii=True)
        logger.info(f"Saved to {output_path}")

    return decompile_pass_rate(testsets, gen_results_repeat, opts, cfg["num_workers"])


In [97]:
import urllib.request
import json

TESTSET_PATH = "/content/decompile_eval.json"

URL = (
    "https://raw.githubusercontent.com/albertan017/LLM4Decompile/main/"
    "legacy-test/decompile-eval-executable-gcc-obj.json"
)

print("Downloading testset...")
urllib.request.urlretrieve(URL, TESTSET_PATH)

with open(TESTSET_PATH) as f:
    data = json.load(f)

from collections import Counter
print(f"✓ {len(data)} samples loaded")
print(Counter(d['type'] for d in data))
print("Keys:", list(data[0].keys()))


✓ 656 samples loaded
Counter({'O0': 164, 'O1': 164, 'O2': 164, 'O3': 164})
Keys: ['task_id', 'type', 'c_func', 'c_test', 'input_asm_prompt']


In [98]:
results = run_eval_pipeline(CONFIG)

2026-03-12T12:54:55.489059+0000 | run_eval_pipeline | INFO - Loaded 656 test cases
2026-03-12T12:54:55.490592+0000 | run_eval_pipeline | INFO - Loading model: /content/models/Devstral-Small-2-24B
INFO 03-12 12:54:55 [utils.py:238] non-default args: {'max_model_len': 8192, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'model': '/content/models/Devstral-Small-2-24B'}
INFO 03-12 12:54:55 [model.py:531] Resolved architecture: PixtralForConditionalGeneration
INFO 03-12 12:54:55 [model.py:1554] Using max model len 8192
INFO 03-12 12:54:55 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 03-12 12:55:31 [llm.py:388] Supported tasks: ['generate']
2026-03-12T12:55:31.379096+0000 | run_eval_pipeline | INFO - Generation pass 1/1


Rendering prompts:   0%|          | 0/656 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/656 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

2026-03-12T12:56:16.900860+0000 | run_eval_pipeline | INFO - Saved to ./output.json


Tallying: 100%|██████████| 656/656 [00:00<00:00, 1657508.09it/s]


  Opt O0: Compile 23.78%  |  Run 7.93%
  Opt O1: Compile 14.02%  |  Run 4.27%
  Opt O2: Compile 21.95%  |  Run 3.05%
  Opt O3: Compile 7.32%  |  Run 2.44%


In [99]:
from google.colab import files
files.download('./output.json')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>